#Train Model & View in TensorBoard

Optional notebook for model training as an alternative to using the CLI train.py

In [ ]:

# config for training
DATA_GLOB   = "../raw_data/*.csv"  # e.g., "raw_data/*training*.csv"
BATCH_SIZE  = 1024
VAL_RATIO   = 0.20
SEED        = 88
GROUP_COL   = "h3_r5"       #can change r7 to different resolution like r5, r6.

# Early stopping objective
MONITOR     = "val_auprc"
PATIENCE    = 10
MIN_DELTA   = 1e-5
MAX_EPOCHS  = 100

# Model hyperparameters
HIDDEN       = 256
DEPTH        = 2
DROPOUT      = 0.05
ACT          = "gelu"           # relu, silu, or gelu
LR           = 2.427e-4 
WEIGHT_DECAY = 9.208e-5
STANDARDIZE  = False            # do not need to standardize for alphaearth data

# where to send logs
LOG_ROOT = "../outputs/logs"


In [ ]:

from pathlib import Path
import glob
import pandas as pd
import numpy as np
import torch

from irr.training.train import run_train
from irr.configs import TrainConfig
from irr.models.mlp_classifier import ModelConfig

# find newest best.ckpt
def find_latest_best_ckpt(root="outputs/logs"):
    pattern = str(Path(root) / "**" / "checkpoints" / "best.ckpt")
    paths = glob.glob(pattern, recursive=True)
    if not paths:
        return None
    paths.sort(key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return paths[0]

# read last-epoch metrics from a CSV logger
def read_last_epoch_metrics(csv_log_dir: str) -> pd.Series | None:
    metrics_csv = Path(csv_log_dir) / "metrics.csv"
    if not metrics_csv.exists():
        return None
    m = pd.read_csv(metrics_csv)
    if "epoch" not in m.columns and "step" in m.columns:
        m["epoch"] = m["step"]
    m = m.sort_values(["epoch"]).groupby("epoch").last()
    return m.iloc[-1] if len(m) else None


In [3]:

# Build model config and training config
group_col = None if str(GROUP_COL).lower() == "none" else GROUP_COL

model_cfg = ModelConfig(
    hidden=HIDDEN,
    depth=DEPTH,
    dropout=DROPOUT,
    act=ACT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    standardize=STANDARDIZE,
)

cfg = TrainConfig(
    data_glob=DATA_GLOB,
    batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO,
    seed=SEED,
    monitor=MONITOR,
    patience=PATIENCE,
    max_epochs=MAX_EPOCHS,
    model=model_cfg,
    group_col=group_col
    # min_delta was removed
    # min_delta=MIN_DELTA,
)

print("Training with config:")
print(cfg)

result = run_train(cfg)
csv_log_dir = Path(result.get("log_dir", "outputs/logs"))
print(f"CSV logger directory: {csv_log_dir}")


Training with config:
TrainConfig(data_glob='../raw_data/*.csv', batch_size=1024, val_ratio=0.2, seed=88, include_states=None, group_col='h3_r7', monitor='val_auprc', patience=10, min_delta=1e-05, max_epochs=100, model=ModelConfig(in_dim=64, hidden=256, depth=2, dropout=0.1, act='silu', lr=0.001, weight_decay=0.0001, standardize=False), hidden=256, depth=2, dropout=0.1, act='silu', lr=0.001, weight_decay=0.0001, standardize=False)
[Split] grouped by 'h3_r7'. train=311,194  val=77,910


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[Group check] train_groups=136,747  val_groups=34,187  overlap=0
[Split] group_col='h3_r7'



  | Name        | Type                   | Params | Mode 
---------------------------------------------------------------
0 | net         | MLPHead                | 149 K  | train
1 | loss        | BCEWithLogitsLoss      | 0      | train
2 | train_auroc | BinaryAUROC            | 0      | train
3 | train_auprc | BinaryAveragePrecision | 0      | train
4 | val_auroc   | BinaryAUROC            | 0      | train
5 | val_auprc   | BinaryAveragePrecision | 0      | train
6 | val_cm      | BinaryConfusionMatrix  | 0      | train
---------------------------------------------------------------
149 K     Trainable params
0         Non-trainable params
149 K     Total params
0.598     Total estimated model params size (MB)
21        Modules in train mode
0         Modules in eval mode


[Split] predefined indices. train=311,194  val=77,910
Epoch 1:   0%|          | 0/304 [00:00<?, ?it/s, v_num=1, val_loss=0.176, val_auroc=0.000, val_auprc=-1.66e+4, train_loss=0.219]          

...

Epoch 18:   4%|▎         | 11/304 [00:00<00:04, 68.46it/s, v_num=1, val_loss=0.157, val_auroc=0.000, val_auprc=-1.17e+5, train_loss=0.127, train_auroc=0.000, train_auprc=nan.0] 

/Users/tim/repos/pytorch_irr_model_modular/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)
/Users/tim/repos/pytorch_irr_model_modular/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


Epoch 18: 100%|██████████| 304/304 [00:05<00:00, 56.57it/s, v_num=1, val_loss=0.151, val_auroc=0.000, val_auprc=1.4e+3, train_loss=0.126, train_auroc=0.000, train_auprc=nan.0]  


/Users/tim/repos/pytorch_irr_model_modular/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)
/Users/tim/repos/pytorch_irr_model_modular/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


CSV logger directory: outputs/logs/mlp_classifier/version_1


In [ ]:
last = read_last_epoch_metrics(str(csv_log_dir))
if last is None:
    print("metrics.csv not found yet in CSV logger directory.")
else:
    display(last.to_frame().T)
    keys = ["val_auprc", "val_auroc", "train_loss", "val_loss"]
    print({k: float(last.get(k)) for k in keys if k in last.index})


,step,train_auprc,train_auroc,train_loss,val_auprc,val_auroc,val_loss
18,5775.0,NaN,0.0,0.126302,1396.956177,0.0,0.151057


{'val_auprc': 1396.9561767578125, 'val_auroc': 0.0, 'train_loss': 0.1263020187616348, 'val_loss': 0.15105701982975}


In [ ]:
ckpt = find_latest_best_ckpt(LOG_ROOT)
if ckpt is None:
    print("Could not find best.ckpt under", LOG_ROOT)
else:
    print("Best checkpoint:", ckpt)

Best checkpoint: outputs/logs/mlp_classifier_tb/version_1/checkpoints/best.ckpt


## View in TensorBoard

From your project root, run:

```bash
poetry run tensorboard --logdir ../outputs/logs --port 6006
```

Then open http://localhost:6006/ in your browser.
